# Estado do Agente

No notebook anterior, vimos que o **contexto** permite injetar informacoes externas no agente. Mas o contexto e read-only: as tools podem ler, mas nao podem modificar. Em muitas situacoes, precisamos que o agente **extraia e armazene** informacoes ao longo da conversa.

Por exemplo: o usuario menciona que quer um apartamento no Leblon em uma mensagem, e na proxima mensagem diz que o orcamento e de 1.2 milhao. O agente precisa acumular essas informacoes de forma estruturada. Esse e o papel do **estado** (`state_schema`).

A diferenca fundamental:

- **Contexto** -- read-only, definido externamente, nao muda durante a conversa
- **Estado** -- read-write, mutado pelo agente via tools, persiste entre turnos

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Definindo o estado

O estado e definido como uma classe que herda de `AgentState`. Cada campo representa uma informacao que o agente pode ler e escrever ao longo da conversa.

In [ ]:
from langchain.agents import AgentState

class EstadoBusca(AgentState):
    cidade: str
    bairro: str
    orcamento: float
    tipo_imovel: str

## Escrevendo no estado

Para que uma tool escreva no estado, ela retorna um `Command(update={...})`. Alem de atualizar os campos do estado, a tool precisa incluir uma `ToolMessage` na lista de `messages`. Isso e necessario porque o modelo espera receber uma resposta para cada tool call que ele faz.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command

@tool
def atualizar_busca(cidade: str, bairro: str, orcamento: float, tipo_imovel: str, runtime: ToolRuntime) -> Command:
    """Atualiza os criterios de busca do cliente quando ele informar suas preferencias."""
    return Command(update={
        "cidade": cidade,
        "bairro": bairro,
        "orcamento": orcamento,
        "tipo_imovel": tipo_imovel,
        "messages": [ToolMessage("Criterios de busca atualizados.", tool_call_id=runtime.tool_call_id)]
    })

O `Command` e o mecanismo que permite a tool modificar o estado do agente. O campo `update` recebe um dicionario com os campos a serem atualizados. A `ToolMessage` e obrigatoria para que o modelo saiba que a tool foi executada com sucesso.

## Agente com estado

Agora criamos o agente passando `state_schema` e um `checkpointer` para persistir o estado entre turnos da conversa.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agente = create_agent(
    model="gpt-4.1-nano",
    tools=[atualizar_busca],
    checkpointer=InMemorySaver(),
    state_schema=EstadoBusca
)

config = {"configurable": {"thread_id": "busca-1"}}

## Testando a escrita no estado

Quando o usuario menciona suas preferencias, o agente deve extrair as informacoes e gravar no estado usando a tool `atualizar_busca`.

In [ ]:
from langchain.messages import HumanMessage

resposta = agente.invoke(
    {"messages": [HumanMessage(content="Quero um apartamento no Leblon, Rio de Janeiro, orcamento de 1.2 milhao.")]},
    config
)

print(resposta["messages"][-1].content)

Vamos verificar se o estado foi realmente atualizado inspecionando as mensagens.

In [ ]:
from pprint import pprint

pprint(resposta["messages"])

## Lendo o estado

Para que o agente consulte os dados gravados no estado, criamos uma tool de leitura que acessa `runtime.state`. Vamos adicionar essa tool e recriar o agente.

In [ ]:
@tool
def consultar_criterios(runtime: ToolRuntime) -> str:
    """Consulta os criterios de busca do cliente armazenados no estado."""
    cidade = runtime.state.get("cidade", "nao definido")
    bairro = runtime.state.get("bairro", "nao definido")
    orcamento = runtime.state.get("orcamento", "nao definido")
    tipo_imovel = runtime.state.get("tipo_imovel", "nao definido")
    return f"Cidade: {cidade}, Bairro: {bairro}, Orcamento: {orcamento}, Tipo: {tipo_imovel}"

In [ ]:
agente = create_agent(
    model="gpt-4.1-nano",
    tools=[atualizar_busca, consultar_criterios],
    checkpointer=InMemorySaver(),
    state_schema=EstadoBusca
)

config = {"configurable": {"thread_id": "busca-2"}}

## Conversa em dois turnos

Agora vamos testar o fluxo completo: no primeiro turno o usuario define suas preferencias, e no segundo turno pergunta quais sao os criterios armazenados. O estado deve persistir entre os turnos porque estamos usando o mesmo `thread_id`.

In [ ]:
resposta = agente.invoke(
    {"messages": [HumanMessage(content="Estou procurando uma casa em Jardim Botanico, Rio de Janeiro, ate 900 mil.")]},
    config
)

print(resposta["messages"][-1].content)

In [ ]:
resposta = agente.invoke(
    {"messages": [HumanMessage(content="Quais sao os meus criterios de busca?")]},
    config
)

print(resposta["messages"][-1].content)

O agente lembrou dos criterios definidos no turno anterior. Diferente da memoria (que guarda mensagens), o estado guarda **dados estruturados** que as tools podem ler e escrever programaticamente. Isso e muito mais confiavel do que depender do modelo para extrair informacoes do historico de mensagens.

No proximo notebook, vamos combinar o que aprendemos sobre tools, contexto e estado para construir sistemas com **multiplos agentes especializados**.